# Catalog Data Pipeline: Web Scraping to Normalized SQLite
### Zepto Capstone Project ? Module 1

This notebook builds the data pipeline for scraping catalog items, cleaning types, converting currency, storing data into a normalized SQLite database, and querying it.

In [1]:
import os
import sqlite3
import pandas as pd
from pipeline import scrape_category, CATEGORIES_TO_SCRAPE

all_books = []
for cat_name, rel_url in CATEGORIES_TO_SCRAPE:
    print(f"Scraping category: {cat_name}...")
    items = scrape_category(cat_name, rel_url)
    print(f"  Fetched {len(items)} items.")
    all_books.extend(items)

print(f"Total raw books scraped: {len(all_books)}")

Scraping category: Travel...
  Fetched 11 items.
Scraping category: Mystery...
  Fetched 32 items.
Scraping category: Historical Fiction...
  Fetched 26 items.
Scraping category: Sequential Art...
  Fetched 75 items.
Total raw books scraped: 144


## Data Cleaning & Currency Conversion
- Extract numeric float `price_gbp`.
- Convert text ratings (`One`..`Five`) to integers 1?5.
- Parse availability string into boolean `in_stock`.
- Convert to INR using the fixed project baseline: **1 GBP = 105.50 INR**.

In [2]:
from pipeline import clean_data

cleaned_df = clean_data(all_books)
print(cleaned_df.head())

Total raw books scraped: 144
Cleaned dataset shape: (144, 6)
                                 title  price_gbp  price_inr  rating  in_stock        category
0   1,000 Places to See Before You Die      26.08    2751.44       5         1          Travel
1                 It's Only the Himalayas      45.17    4765.44       2         1          Travel
2  Slow States of West Africa           33.66    3551.13       3         1          Travel
3     Full Bore (Kevin Kerney #3)       44.34    4677.87       2         1         Mystery
4                 A Flight of Arrows    55.53    5858.42       5         1  Historical Fiction


## Normalized SQLite Schema & Population
We populate two tables in `zepto_catalog.db`: `categories` and `books` with a foreign key relationship.

In [3]:
from pipeline import init_db, load_to_sqlite, DB_PATH

conn = init_db(DB_PATH)
load_to_sqlite(cleaned_df, conn)

Successfully loaded 144 books across 4 categories into SQLite.


## Executing Analytical SQL Queries
Running 5 queries demonstrating `WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `BETWEEN`, `IN`, and `JOIN`.

In [4]:
print("--- Top 5 Most Expensive Books (INR) ---")
q2 = pd.read_sql("SELECT title, price_inr, rating FROM books ORDER BY price_inr DESC LIMIT 5", conn)
print(q2.to_string(index=False))

print("\n--- Top 5 Books with JOIN to Categories ---")
q5 = pd.read_sql("""
    SELECT b.book_id, b.title, c.category_name, b.price_gbp, b.price_inr, b.rating
    FROM books b
    JOIN categories c ON b.category_id = c.category_id
    WHERE b.rating = 5
    ORDER BY b.price_inr DESC
    LIMIT 5
""", conn)
print(q5.to_string(index=False))

--- Top 5 Most Expensive Books (INR) ---
                                                                 title  price_inr  rating
                                         Boar Island (Anna Pigeon #19)    6275.14       3
The No. 1 Ladies' Detective Agency (No. 1 Ladies' Detective Agency #1)    6087.35       4
                                                              El Deafo    6078.91       5
                      Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)    6019.83       4
                                      A Year in Provence (Provence #1)    6000.84       4

--- Top 5 Books with JOIN to Categories ---
 book_id                                                                 title   category_name  price_gbp  price_inr  rating
     104                                                              El Deafo  Sequential Art      57.62    6078.91       5
     118         The Sandman, Vol. 3: Dream Country (The Sandman #3)            Sequential Art      55.55    5860.52       5

## Verifying SQL JOIN vs. Pandas in-Memory Merge Equivalence

In [5]:
from pipeline import verify_pandas_equivalence
print("Checking equivalence between pd.read_sql and pd.merge...")
df_sql, df_merge = verify_pandas_equivalence(conn)
conn.close()

Checking equivalence between pd.read_sql and pd.merge...
SUCCESS: Both DataFrames match identically!
